# Copying necessary steps from data_loading.py

In [938]:
## creating variable to trigger different components of script based on what you want to run
cad_all = False
cad_crude = True

In [939]:
import pandas as pd
from pathlib import Path
from ydata_profiling import ProfileReport
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from importlib import reload
import data_loading  # Import the module instead of specific functions


reload(data_loading)  # Reload the module after making changes

# Now you can reference the functions directly from the reloaded module
read_csv_to_dataframe = data_loading.read_csv_to_dataframe
read_txt_to_dataframe = data_loading.read_txt_to_dataframe

In [940]:
"""
Expands the categories in a given column of a DataFrame into separate binary columns.

Parameters:
- df: pandas.DataFrame, the DataFrame to modify.
- column_name: str, the name of the column to expand.
"""
def expand_categories_in_column(df, column_name):

    unique_categories = set()
    df[column_name].dropna().apply(lambda x: unique_categories.update(set(x.split(', '))))

    # init dict to hold new columns
    new_columns = {}
    
    for category in unique_categories:
        # Ensure category name is valid as a column name (e.g., no spaces or special characters)
        # add col name as there are duplicates across different cols
        valid_category_name = column_name + " / " + category.lower().replace(' ', '_').replace(',', '')
        
        # Instead of modifying df directly, create and store the new column in new_columns
        mask = df[column_name].fillna('').str.contains(category, regex=False, na=False)
        new_columns[valid_category_name] = mask.astype(int)

    # Create a new DataFrame from the new_columns dictionary
    new_columns_df = pd.DataFrame(new_columns, index=df.index)
    
    # Concatenate the new columns to the original DataFrame
    df = pd.concat([df, new_columns_df], axis=1)

    

    return df

In [941]:
## making file path imports more robust

# get directory of current file
current_script_directory = Path.cwd()

# Construct path to data files given relative location
cad_string = current_script_directory / "../data/raw/canada/"
usa_string = current_script_directory  / "../data/raw/usa/"

cad_data = cad_string / "pipeline-incidents-comprehensive-data.csv"
cad_unknown_data = cad_string / "PODSdb_MDOTW_VW_OCCURRENCE_PUBLIC.csv"

usa_pre1986 = usa_string / "accident_hazardous_liquid_pre1986/accident_hazardous_liquid_pre1986.txt"
usa_1986_jan2002 = usa_string / "accident_hazardous_liquid_1986_jan2002/accident_hazardous_liquid_1986_jan2002.txt"
usa_jan2002_dec2009 = usa_string / "accident_hazardous_liquid_jan2002_dec2009/accident_hazardous_liquid_jan2002_dec2009.txt"
usa_jan2010_present = usa_string / "accident_hazardous_liquid_jan2010_present/accident_hazardous_liquid_jan2010_present.txt"
usa_gravity = usa_string / "accident_gravity_reporting_regulated_jul2020_present/accident_gravity_reporting_regulated_jul2020_present.txt"

# read data into dataframes
CAD_data_raw = read_csv_to_dataframe(cad_data)
CAD_unknown_data_raw = read_csv_to_dataframe(cad_unknown_data)

usa_pre1986_raw = read_txt_to_dataframe(usa_pre1986)
usa_1986_jan2002_raw = read_txt_to_dataframe(usa_1986_jan2002)
usa_jan2002_dec2009_raw = read_txt_to_dataframe(usa_jan2002_dec2009)
usa_jan2010_present_raw = read_txt_to_dataframe(usa_jan2010_present)
usa_gravity_raw = read_txt_to_dataframe(usa_gravity)

File at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/canada/pipeline-incidents-comprehensive-data.csv' successfully read into a DataFrame.
File at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/canada/PODSdb_MDOTW_VW_OCCURRENCE_PUBLIC.csv' successfully read into a DataFrame.
TXT file at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/usa/accident_hazardous_liquid_pre1986/accident_hazardous_liquid_pre1986.txt' successfully read into a DataFrame.
TXT file at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/usa/accident_hazardous_liquid_1986_jan2002/accident_hazardous_liquid_1986_jan2002.txt' successfully read into a DataFrame.
TXT file at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/usa/accident_hazardous_l

# Cleaning

## CAD_data_raw

### All subtances

In [942]:
# make deep copy for cleaned df
CAD_data_cleaned = CAD_data_raw.copy()

# Generate profiling report to identify outliers/trends etc. for cleansing purposes
# CAD_profile = ProfileReport(CAD_data_raw, title="CAD data raw Profiling Report", explorative=True)
# CAD_profile.to_file("CAD_report.html")

# report generated that there are no duplicate rows and that country column has constant value but it's going to be dropped below anyway

In [943]:
## CORROSION only mentioned in "detailed what happened" and "what happened category"

# looked at data dictionary for columns that could be relevant to substance/pipeline specifications/cause of incident
# want to analyze this subset for trends/correlations
## data dict and data don't line up so needed to manually change some; labelled with "## different"
subset_columns = [
"pipeline or facility type",
"pipeline or facility equipment involved",
"rupture",
"incident types", ## different
"conditions that resulted in the operation beyond limits",
"pipeline outside diameter (nps)",
"pipeline length (km)",
"substance carried",
"released substance type",
"facility type", ## different
"facility latitude",
"facility longitude",
"longitude",
"latitude",
"nominal pipe size",
"material",
"material grade",
"schedule",
"design wall thickness (mm)",
"custom design wall thickness (mm)",
"actual wall thickness (mm)",
"licensed maximum operating pressure (kpa)",
"actual operating pressure at time of failure (kpa)",
"year of manufacture",
"most recent cathodic protection reading at incident site (mv vs. cu/cuso4)",
"weld type",
"seam type",
"coating location",
"coating type",
"coating condition",
"application method",
"year when the coating was applied",
"insulation installed",
"detailed what happened", ## diff
"what happened category", ## diff
"detailed why it happened", ## diff
"why it happened category" ## diff
]

# Taking the subset
CAD_data_cleaned = CAD_data_cleaned[subset_columns]

# only going to take first of pipeline outside diameter values
CAD_data_cleaned['pipeline outside diameter (nps)'] = CAD_data_cleaned['pipeline outside diameter (nps)'].str.split(',').str[0].astype(float).fillna(0)

# fill empty length values with 0
CAD_data_cleaned['pipeline length (km)'] = CAD_data_cleaned['pipeline length (km)'].astype(float).fillna(0)

# removing inches and keeping in mm
CAD_data_cleaned['design wall thickness (mm)'] = CAD_data_cleaned['design wall thickness (mm)'].str.split(' mm').str[0].astype(float).fillna(0)

## 3 substance-based columns - if they're all empty/NA then we'll drop the rows as these don't really help us
## EDIT: "substance" and "released substance type" are identical cols so removed substance
CAD_data_cleaned = CAD_data_cleaned.loc[~((CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].isna()))]

# # drop columns that have more than x percentage of na's
# # can play around with this value to see how it affects results
# threshold = 0.5
# CAD_data_cleaned = CAD_data_cleaned.loc[:, CAD_data_cleaned.isnull().mean() < 0.5]

# #categorical_cols = CAD_data_cleaned.select_dtypes(include=['object']).columns
# #categorical_cols

In [944]:
## lots of na's in substance carried col - try to imputate data as best as we can

## OPTION 1: substance carried contains crude oil, released substance does not; released substance = lube oil, drilling fluid, natural gas liquids, diesel fuel, condensate
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil')==False)][['released substance type', 'substance carried']].drop_duplicates()


,released substance type,substance carried
153,Lube Oil,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."
232,Drilling Fluid,Crude Oil
284,Natural Gas Liquids,Crude Oil
349,Natural Gas Liquids,"Condensate, Crude Oil, Natural Gas Liquids"
593,Drilling Fluid,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."
1050,Natural Gas - Sweet,Crude Oil
1160,Diesel Fuel,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."
1487,Condensate,Crude Oil
1838,Hydraulic Fluid,Crude Oil


In [945]:
## OPTION 2: substance carried contains crude oil, released subsance does too
## OBSERVATION: released substance type sometimes is more specific (sour vs. sweet) than substance carried
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil'))][['released substance type', 'substance carried']].drop_duplicates()


,released substance type,substance carried
10,Crude Oil - Sour,Crude Oil
29,Crude Oil - Sweet,Crude Oil
183,Crude Oil - Synthetic,Crude Oil
196,Crude Oil - Sweet,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."


In [946]:
## OPTION 3: substance carried doesn't contain crude oil, released substance does? 
## NO RESULTS
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')==False) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil'))][['released substance type', 'substance carried']].drop_duplicates()

,released substance type,substance carried


In [947]:
## OPTION 4: substance carried doesn't contain crude oil, released substance doesn't either
## HAPPENS A LOT - and the released substance type looks similar as if the substance carried was crude oil
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')==False) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil')==False)][['released substance type', 'substance carried']].drop_duplicates()

,released substance type,substance carried
4,Natural Gas - Sweet,Natural Gas
14,Jet Fuel,"not applicable, Refined Products-Aviation, Ref..."
16,Natural Gas - Sweet,"Natural Gas, Natural Gas Sweet"
17,Water,White Water
31,Natural Gas - Sour,Natural Gas Sour
79,Natural Gas - Sweet,Natural Gas Sweet
87,Pulp slurry,Sulfite Pulp Slurry
101,Mixed HVP Hydrocarbons,Natural Gas Sweet
281,Contaminated Water,"Natural Gas Sour, Natural Gas Sweet, not appli..."
282,Propane,Natural Gas Sour


In [948]:
# option 5 - substance carried is na, but released substance contains crude oil
## Can confidently change the substance carried to crude oil based on results from option #3
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil'))][['released substance type', 'substance carried']].drop_duplicates()
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil')), 'substance carried'] = 'Crude Oil'

In [949]:
# option 6 - substance carried is na, released substance does not contain crude oil
## change substance carried to NOT crude oil IFF the released substance type has 0% change of appearing in crude oil (source of list: chatgpt)
not_in_pipeline_crude_oil = [
    "Potassium Hydroxide (caustic solution)",
    "Sulphur Dioxide",
    "Water",
    "Potassium Carbonate",
    "Contaminated Water",
    "Waste Oil",
    "Amine",
    "Produced Water",
    "Glycol",
    "Pulp slurry"
]
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil')==False)][['released substance type', 'substance carried']].drop_duplicates()
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].isin(not_in_pipeline_crude_oil)), 'substance carried'] = 'NOT crude oil'



In [950]:
# OPTION 7/8 - released substance type is na; can't assume what was released (if anything); ignore! 
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')) & (CAD_data_cleaned['released substance type'].isna())][['released substance type', 'substance carried']].drop_duplicates()

,released substance type,substance carried
22,NaN,Crude Oil
24,NaN,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."
1732,NaN,"Condensate, Crude Oil"


In [951]:
# still left with lots of records where substance carried is N/A
# sadly have to drop these as it's going to be our target/key attribute and we cannot imputate any further
CAD_data_cleaned = CAD_data_cleaned.loc[CAD_data_cleaned['substance carried'].isna()==False]

In [952]:
if cad_all:
    # map all non-crude oil substances into one group to see if there's a difference
    CAD_data_cleaned.loc[CAD_data_cleaned['substance carried'].astype(str).str.contains('Crude Oil')==False, 'substance carried'] = "NOT Crude Oil"

    ## FOR argument's sake, let's also combine the crude oils
    CAD_data_cleaned.loc[CAD_data_cleaned['substance carried'].astype(str).str.contains("NOT Crude Oil")==False, 'substance carried'] = "Crude Oil"

# Exploring Data

In [953]:
# SPLIT df into substance carried contains crude oil or does not
crude_cad = CAD_data_cleaned.copy()
crude_cad = crude_cad.loc[crude_cad['substance carried'] == "Crude Oil"]

non_crude_cad = CAD_data_cleaned.copy()
non_crude_cad = non_crude_cad.loc[non_crude_cad['substance carried'] == "NOT Crude Oil"]

In [954]:
non_crude_cad.describe()

,pipeline outside diameter (nps),pipeline length (km),facility latitude,facility longitude,longitude,latitude,schedule,design wall thickness (mm),custom design wall thickness (mm),actual wall thickness (mm),licensed maximum operating pressure (kpa),actual operating pressure at time of failure (kpa),year of manufacture,most recent cathodic protection reading at incident site (mv vs. cu/cuso4),year when the coating was applied
count,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [955]:
# OBSERVATION: large difference between licensend maximum kpa and actual operating kpa at time of failure for crude
# different in non crude is small. but this is  SMALL sample size
crude_cad.describe()

,pipeline outside diameter (nps),pipeline length (km),facility latitude,facility longitude,longitude,latitude,schedule,design wall thickness (mm),custom design wall thickness (mm),actual wall thickness (mm),licensed maximum operating pressure (kpa),actual operating pressure at time of failure (kpa),year of manufacture,most recent cathodic protection reading at incident site (mv vs. cu/cuso4),year when the coating was applied
count,356.000000,356.000000,83.000000,83.000000,356.000000,356.000000,8.000000,356.000000,2.000000,9.000000,34.000000,16.000000,18.000000,6.000000,10.000000
mean,715.629775,880.269365,50.690117,-113.465640,-112.602278,50.935043,90.000000,0.452809,7.550000,8.022222,5853.408824,1713.909375,1984.166667,320.034500,1972.900000
std,348.111596,524.470741,2.817383,12.216978,11.849135,2.945281,50.142654,1.879802,2.757716,2.339219,12896.902093,1707.373118,26.747732,1401.031221,21.429212
min,0.000000,0.000000,42.952615,-122.950276,-123.702550,42.834210,40.000000,0.000000,5.600000,5.000000,0.000000,0.000000,1950.000000,-1109.000000,1950.000000
25%,508.000000,924.544450,49.268489,-122.202148,-121.339343,49.262374,40.000000,0.000000,6.575000,7.100000,0.000000,62.025000,1958.250000,-1.353500,1958.250000
50%,762.000000,946.159155,50.662609,-116.647554,-113.362323,50.643461,90.000000,0.000000,7.550000,7.300000,1898.000000,1335.500000,1984.000000,-1.225000,1967.500000
75%,762.000000,1248.909872,52.645367,-111.273003,-110.496198,53.461852,115.000000,0.000000,8.525000,9.500000,7655.000000,2972.962500,2012.000000,-0.266250,1979.500000
max,1219.000000,2334.948756,59.066738,-72.627000,-72.478000,63.668419,160.000000,12.700000,9.500000,12.400000,75403.000000,5063.000000,2020.000000,3033.000000,2014.000000


In [956]:
## Frequency Counts for categorical attributes (excluding the 'substance carried' column)
frequency_counts_crude = crude_cad.select_dtypes(include=['object']).drop(columns=['substance carried']).apply(pd.Series.value_counts)
frequency_counts_not_crude = non_crude_cad.select_dtypes(include=['object']).drop(columns=['substance carried']).apply(pd.Series.value_counts)

In [957]:
for col in frequency_counts_not_crude.columns:
    print(col, "\n")
    print(frequency_counts_not_crude.loc[frequency_counts_not_crude[col].isna()==False][col])
    print("\n")

pipeline or facility type 

Series([], Name: pipeline or facility type, dtype: object)


pipeline or facility equipment involved 

Series([], Name: pipeline or facility equipment involved, dtype: object)


rupture 

Series([], Name: rupture, dtype: object)


incident types 

Series([], Name: incident types, dtype: object)


conditions that resulted in the operation beyond limits 

Series([], Name: conditions that resulted in the operation beyond limits, dtype: object)


released substance type 

Series([], Name: released substance type, dtype: object)


facility type 

Series([], Name: facility type, dtype: object)


nominal pipe size 

Series([], Name: nominal pipe size, dtype: object)


material 

Series([], Name: material, dtype: object)


material grade 

Series([], Name: material grade, dtype: object)


weld type 

Series([], Name: weld type, dtype: object)


seam type 

Series([], Name: seam type, dtype: object)


coating location 

Series([], Name: coating location, dtype: objec

In [958]:
for col in frequency_counts_crude.columns:
    print(col, "\n")
    print(frequency_counts_crude.loc[frequency_counts_crude[col].isna()==False][col])
    print("\n")

pipeline or facility type 

Distribution     36.0
Gathering         4.0
Processing        2.0
Transmission    227.0
Name: pipeline or facility type, dtype: float64


pipeline or facility equipment involved 

No     221.0
Yes    135.0
Name: pipeline or facility equipment involved, dtype: float64


rupture 

No     354.0
Yes      2.0
Name: rupture, dtype: float64


incident types 

Adverse Environmental Effects                           41.0
Explosion                                                6.0
Explosion, Fire                                          1.0
Fatality                                                 2.0
Fire                                                   100.0
Operation Beyond Design Limits                          77.0
Release of Substance                                    75.0
Release of Substance, Adverse Environmental Effects      3.0
Serious Injury (CER or TSB)                             51.0
Name: incident types, dtype: float64


conditions that resulted in t

## Logistic Regression

In [959]:
### trying to map qualitative comments (why it happened, etc.) to crude oil vs. non crude oil to see if there are any trends
# start by only taking relevant columns
test_df = CAD_data_cleaned.copy()
# test_df = test_df[['substance carried', 'what happened category', 'detailed what happened',
#     'detailed why it happened', 'why it happened category',
#     'incident types']]
test_df = test_df[['substance carried', 'what happened category', 'detailed what happened', 'incident types']]

In [960]:
## went through for words related to chemistry - temperatures, corrosion, cracking, etc.
## filter first for what columns that contain these, make sure lower case
test_df['what happened category'] = test_df['what happened category'].str.lower()
test_df['detailed what happened'] = test_df['detailed what happened'].str.lower()
chem_words = ['corrosion', 'temperature', 'deterioration', 'overheating', 'weather', 'frost', 'fire', 'frozen', 'chemical', 'cracking']

In [961]:
def contains_chem_words(text, chem_words):
    if pd.isna(text):
        return False
    return any(chem_word in text for chem_word in chem_words)

# Create a boolean mask where at least one of the conditions is True
mask = test_df.apply(lambda row: contains_chem_words(row['what happened category'], chem_words) or 
                                 contains_chem_words(row['detailed what happened'], chem_words), axis=1)

# Filter the DataFrame
filtered_df = test_df[mask]

In [962]:
filtered_df.count()
test_df = filtered_df.copy()
test_df.count()

substance carried         809
what happened category    809
detailed what happened    809
incident types            809
dtype: int64

In [963]:
## expand categorical cols with multiple options into unique binary cols
## e.g., incident types: "fire", "fire, release of substance"
# if there are 5 of these options, it means 15 possible combinations for 1-hot encoding, instead of just 5 for binary
# columns_to_expand = [
#     'what happened category',
#     'detailed what happened',
#     'detailed why it happened',
#     'why it happened category',
#     'incident types'
# ]

columns_to_expand = [
    'what happened category',
    'detailed what happened',
    'incident types'
]

for column in columns_to_expand:
    test_df = expand_categories_in_column(test_df, column)

test_df.drop(columns=columns_to_expand, inplace=True)

test_df['substance carried'] = test_df['substance carried'].map({'NOT Crude Oil': 0, 'Crude Oil': 1})

In [964]:
## make corr matrix and find pairs >0.8 (and less than 1)
corr_matrix = test_df.corr()
high_corr_pairs = corr_matrix.unstack().sort_values(kind="quicksort", ascending=False)
high_corr_pairs = high_corr_pairs[(abs(high_corr_pairs) > 0.8) & (high_corr_pairs != 1)]

# drop columns with correlations >0.8
threshold = 0.8
to_drop = set()

for (col1, col2), corr in high_corr_pairs.items():
    if corr > threshold:
        # drop col with less entries
        sum_col1 = test_df[col1].sum()
        sum_col2 = test_df[col2].sum()
        if sum_col1 < sum_col2:
            to_drop.add(col1)
        else:
            to_drop.add(col2)

# Drop identified columns from the DataFrame
test_df_2 = test_df.copy().drop(columns=list(to_drop))

In [965]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

if cad_all:

    # Features and target variable
    X = test_df_2.drop(['substance carried'], axis=1)  # Drop the original target column to get the features
    y = test_df_2['substance carried']  # Binary encoded target variable

    # Split the data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # Create and fit the logistic regression model
    model = LogisticRegression(max_iter=1000)  # Increased max_iter in case of convergence issues
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Model Evaluation
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))

    # Coefficients
    coefficients = pd.DataFrame(model.coef_.flatten(), index=X.columns, columns=['Coefficient'])
    sorted_coefficients = coefficients.sort_values(by='Coefficient', ascending=False)
    print(sorted_coefficients)

## Crude Only

In [966]:
cad_crude = CAD_data_cleaned.copy()
#cad_crude['substance carried'].unique()

# only want to look at crude oil incidents
cad_crude = cad_crude.loc[(cad_crude['substance carried'].str.contains('Crude Oil')) | (cad_crude['released substance type'].str.contains('Crude Oil'))]

# now let's extract the specifications (sour, sweet, heavy, light) into other columns so they're easier to deal with
## ASSUMPTION: 

#cad_crude[['substance carried', 'released substance type']].drop_duplicates()
cad_crude['substance carried'].unique()

## IT'S EITHER all or nothing... so it actually doesn't help

array(['Crude Oil',
       'Crude Oil, Crude Oil Sour Heavy, Crude Oil Sour Light, Crude Oil Sweet Heavy, Crude Oil Sweet Light',
       'Condensate, Crude Oil, Natural Gas Liquids',
       'Condensate, Crude Oil'], dtype=object)

## unused code

In [ ]:
# want to look at incidents that could have been influenced by the material
# e.g., corrosion/cracking
# hard to determine... leaving this for now
# reasons_df = CAD_data_cleaned.copy()[['Detailed what happened', 'What happened category', 'Detailed why it happened', 'Why it happened category']].drop_duplicates()
# reasons_df['Why it happened category'].unique()

## columns that could be related to material influencing an accident/corrosion

#print(CAD_data_cleaned['Pipeline or Facility Type'].unique())
#CAD_data_cleaned['Rupture'].unique() ## loss of containment and unable to operate
#CAD_data_cleaned['Regulation'].unique()
#print(CAD_data_cleaned['Facility Type'].unique())

In [ ]:
# ## NOT RELATIVE bar charts for all categorical

# import seaborn as sns
# import matplotlib.pyplot as plt

# for column in categorical_cols:  # Assuming 'categorical_columns' is a list of your categorical column names
#     plt.figure(figsize=(10, 5))
#     sns.countplot(x=column, hue='substance carried', data=CAD_data_cleaned)  # Replace 'substance_type' with your actual group column
#     plt.title(f'Comparison of {column}')
#     plt.xticks(rotation=45)
#     plt.tight_layout()
#     plt.show()

In [ ]:
## RELATIVE bar charts

# import seaborn as sns
# import matplotlib.pyplot as plt

# # Iterate over each categorical column to create a normalized bar chart
# for column in categorical_cols:  # Assuming 'categorical_columns' is a list of your categorical column names
#     plt.figure(figsize=(10, 5))
    
#     # Create a normalized count plot
#     ax = sns.countplot(x=column, hue='substance carried', data=CAD_data_cleaned)
    
#     # Get the total number of entries for each substance type to normalize the data
#     total_crude = float(CAD_data_cleaned['substance carried'].value_counts()['Crude Oil'])
#     total_non_crude = float(CAD_data_cleaned['substance carried'].value_counts()['NOT Crude Oil'])
    
#     # Iterate through the patches (rectangles/bars) of the countplot
#     for p in ax.patches:
#         # Get the height of the bar which represents the count
#         height = p.get_height()
#         width = p.get_width()
#         x = p.get_x()
#          # Calculate the percentage based on the x position of the bar
#         if x < width:  # This condition checks if the bar belongs to the first group (e.g., Crude Oil)
#             percentage = '{:.1f}%'.format(100 * height / total_crude)
#         else:
#             percentage = '{:.1f}%'.format(100 * height / total_non_crude)

#         # Annotate the bar with the percentage value
#         ax.text(x + width / 2., height, percentage, ha="center", va='bottom')
    
#     plt.title(f'Relative Frequency of {column}')
#     plt.xticks(rotation=45)
#     plt.tight_layout()
#     plt.show()